# Data Overview — PMU Trot Pipeline
Explores `hippique.duckdb`: programme du jour et 61 jours de données historiques (2025-12-28 → 2026-02-26).

In [1]:
import sys
sys.path.insert(0, '..')

from datetime import date
import duckdb
import pandas as pd
from config.settings import DB_PATH

conn = duckdb.connect(str(DB_PATH), read_only=True)
TODAY = date.today().strftime('%Y%m%d')
print(f'Connected to {DB_PATH}')
print(f'Today: {TODAY}')

Connected to c:\Users\julie\OneDrive\Bureau\hippique-prediction\notebooks\..\data\processed\hippique.duckdb
Today: 20260226


## 0. Vue d'ensemble du dataset

In [2]:
# Full dataset summary across all dates
row = conn.execute("SELECT COUNT(DISTINCT date), MIN(date), MAX(date) FROM races").fetchone()
print(f"Période  : {row[1]} → {row[2]} ({row[0]} jours)")

conn.execute("""
    SELECT
        (SELECT COUNT(*)                                     FROM races)                        AS courses,
        (SELECT COUNT(*)                                     FROM runners)                      AS partants,
        (SELECT COUNT(*) FILTER (WHERE finish_position IS NOT NULL) FROM runners)               AS avec_resultats,
        (SELECT COUNT(*) FILTER (WHERE finish_position IS NULL)     FROM runners)               AS sans_resultats,
        (SELECT COUNT(*)                                     FROM odds)                         AS snapshots_cotes
""").df()

Période  : 20251228 → 20260226 (61 jours)


,courses,partants,avec_resultats,sans_resultats,snapshots_cotes
0,1494,17636,12685,4951,30236


## 1. Programme du jour

In [3]:
races = conn.execute(f"""
    SELECT
        race_id,
        hippodrome,
        discipline,
        strftime(race_datetime, '%H:%M') AS depart,
        distance_metres AS dist,
        field_size      AS partants
    FROM races
    WHERE date = '{TODAY}'
    ORDER BY race_id
""").df()

print(f"{len(races)} trot races today ({TODAY})")
races

33 trot races today (20260226)


,race_id,hippodrome,discipline,depart,dist,partants
0,20260226-R1-C1,CAGNES/MER,TROT_ATTELE,14:01,2925,14
1,20260226-R1-C2,CAGNES/MER,TROT_ATTELE,14:36,2700,11
2,20260226-R1-C3,CAGNES/MER,TROT_ATTELE,15:05,2925,8
3,20260226-R1-C4,CAGNES/MER,TROT_ATTELE,15:40,2925,14
4,20260226-R1-C5,CAGNES/MER,TROT_ATTELE,16:15,2150,11
5,20260226-R1-C6,CAGNES/MER,TROT_ATTELE,16:50,2925,14
6,20260226-R1-C7,CAGNES/MER,TROT_ATTELE,17:25,2925,15
7,20260226-R1-C8,CAGNES/MER,TROT_ATTELE,18:00,2925,12
8,20260226-R2-C1,MONCHENGLADBACH,TROT_ATTELE,11:08,2100,8
9,20260226-R2-C2,MONCHENGLADBACH,TROT_ATTELE,11:35,2100,8


In [4]:
# Races by hippodrome (today)
conn.execute(f"""
    SELECT hippodrome, COUNT(*) AS nb_courses
    FROM races
    WHERE date = '{TODAY}'
    GROUP BY hippodrome
    ORDER BY nb_courses DESC
""").df()

,hippodrome,nb_courses
0,LAVAL,8
1,CAGNES/MER,8
2,LYON LA SOIE,7
3,SON PARDO,5
4,MONCHENGLADBACH,5


## 2. Partants

In [5]:
runners_summary = conn.execute(f"""
    SELECT
        COUNT(*)                                     AS total_runners,
        COUNT(*) FILTER (WHERE scratch)              AS scratched,
        COUNT(*) FILTER (WHERE deferre)              AS deferred,
        COUNT(*) FILTER (WHERE musique IS NOT NULL)  AS with_musique,
        ROUND(AVG(handicap_distance), 1)             AS avg_handicap_m
    FROM runners ru
    JOIN races ra ON ra.race_id = ru.race_id
    WHERE ra.date = '{TODAY}'
""").df()
runners_summary

,total_runners,scratched,deferred,with_musique,avg_handicap_m
0,415,8,219,397,2623.4


In [6]:
# Top jockeys by number of rides today
conn.execute(f"""
    SELECT jockey_name, COUNT(*) AS rides
    FROM runners ru
    JOIN races ra ON ra.race_id = ru.race_id
    WHERE jockey_name IS NOT NULL AND NOT scratch
      AND ra.date = '{TODAY}'
    GROUP BY jockey_name
    ORDER BY rides DESC
    LIMIT 15
""").df()

,jockey_name,rides
0,E. RAFFIN,7
1,TH. BRIAND,7
2,D. BEKAERT,7
3,B. CHAUVE-LAFFAY,6
4,R. DERIEUX,6
5,A. FRONTERA,5
6,J.CH. FERON,5
7,BRUSTEN DIETER,5
8,M. MOTTIER,5
9,NIMCZYK MICHAEL,5


## 3. Cotes

In [7]:
odds_summary = conn.execute(f"""
    SELECT
        odds_type,
        COUNT(*)                    AS nb,
        ROUND(MIN(decimal_odds), 2) AS min_cote,
        ROUND(AVG(decimal_odds), 2) AS avg_cote,
        ROUND(MAX(decimal_odds), 2) AS max_cote
    FROM odds o
    JOIN races ra ON ra.race_id = o.race_id
    WHERE ra.date = '{TODAY}'
    GROUP BY odds_type
""").df()
odds_summary

,odds_type,nb,min_cote,avg_cote,max_cote
0,morning,341,1.1,26.74,248.0
1,final,368,1.1,27.83,343.0


In [8]:
# Odds drift: morning vs final (favourites first, today)
drift = conn.execute(f"""
    WITH morning AS (
        SELECT o.runner_id, o.decimal_odds AS cote_matin
        FROM odds o
        JOIN races ra ON ra.race_id = o.race_id
        WHERE o.odds_type = 'morning' AND ra.date = '{TODAY}'
    ),
    final AS (
        SELECT o.runner_id, o.decimal_odds AS cote_direct
        FROM odds o
        JOIN races ra ON ra.race_id = o.race_id
        WHERE o.odds_type = 'final' AND ra.date = '{TODAY}'
    )
    SELECT
        r.race_id,
        r.horse_name,
        m.cote_matin,
        f.cote_direct,
        ROUND(f.cote_direct - m.cote_matin, 2) AS drift,
        CASE
            WHEN f.cote_direct < m.cote_matin THEN 'raccourci'
            WHEN f.cote_direct > m.cote_matin THEN 'allonge'
            ELSE 'stable'
        END AS tendance
    FROM runners r
    JOIN morning m ON m.runner_id = r.runner_id
    JOIN final   f ON f.runner_id = r.runner_id
    WHERE NOT r.scratch
    ORDER BY cote_direct
    LIMIT 20
""").df()
drift

,race_id,horse_name,cote_matin,cote_direct,drift,tendance
0,20260226-R5-C2,HELIOS PIERJI,1.1,1.1,0.0,stable
1,20260226-R7-C7,LOVE ME DE CERISY,1.1,1.2,0.1,allonge
2,20260226-R2-C5,PIRATE NEWPORT,1.1,1.3,0.2,allonge
3,20260226-R5-C4,INTREPIDE OCCAGNES,8.6,1.4,-7.2,raccourci
4,20260226-R2-C3,NJORD INVICTA (SE),1.1,1.4,0.3,allonge
5,20260226-R2-C1,SLIGHTLY LOVE AS,2.4,1.5,-0.9,raccourci
6,20260226-R1-C3,NICIAS,1.6,1.6,0.0,stable
7,20260226-R2-C2,USTRANAS PIPPO,1.8,1.7,-0.1,raccourci
8,20260226-R1-C5,JAVANAIS DELO,2.4,1.9,-0.5,raccourci
9,20260226-R2-C4,KEEP SMILING,3.7,2.1,-1.6,raccourci


## 4. Zoom sur une course

In [9]:
# Pick the race with the most runners today
top_race = conn.execute(f"""
    SELECT race_id, hippodrome, discipline, distance_metres, field_size
    FROM races
    WHERE date = '{TODAY}'
    ORDER BY field_size DESC
    LIMIT 1
""").fetchone()

race_id = top_race[0]
print(f"Course choisie : {race_id}  ({top_race[1]} — {top_race[2]} — {top_race[3]}m — {top_race[4]} partants)")

Course choisie : 20260226-R4-C2  (LAVAL — TROT_ATTELE — 2850m — 18 partants)


In [10]:
conn.execute("""
    SELECT
        r.horse_number AS n,
        r.horse_name,
        r.jockey_name,
        r.handicap_distance AS hcap,
        r.deferre,
        r.scratch,
        r.musique,
        ROUND(om.decimal_odds, 1) AS cote_matin,
        ROUND(od.decimal_odds, 1) AS cote_direct
    FROM runners r
    LEFT JOIN odds om ON om.runner_id = r.runner_id AND om.odds_type = 'morning'
    LEFT JOIN odds od ON od.runner_id = r.runner_id AND od.odds_type = 'final'
    WHERE r.race_id = ?
    ORDER BY COALESCE(od.decimal_odds, om.decimal_odds)
""", [race_id]).df()

,n,horse_name,jockey_name,hcap,deferre,scratch,musique,cote_matin,cote_direct
0,17,LUCY DU RILER,F. NIVARD,2875,True,False,2a(25)2a4a3a3a5a5a0a(24),4.5,3.5
1,15,LARA VRIE,E. RAFFIN,2875,True,False,1a4a3a(25)Da4a3aDmDm3m,4.1,4.2
2,8,LADY DES SOURCES,PH. DAUGEARD,2850,True,False,4a7a(25)Da4a1a1a,7.2,8.4
3,16,LOLLY STAR,F.P. BOSSUET,2875,True,False,5a2a(25)9a5a4a2a7a(24)3m,10.0,9.8
4,5,L'AGATHE VAUCEENNE,M. ABRIVARD,2850,True,False,2a3a0a(25)3a5a8a4a3a4a,13.0,13.0
5,14,LOUISIANE PARIS,R. LAMY,2875,True,False,4aDa1a(25)8a3a7a4a6aDa,16.0,14.0
6,2,LANKA DE GINAI,F. DESMIGNEUX,2850,True,False,(25)0a7a8a8a3a5a5a1a6a,21.0,23.0
7,13,LOULOUTE DE BOUERE,S. POILANE,2875,False,False,0a(25)7aDa1a6a5a0a5a6a,18.0,24.0
8,18,LUANA SPORT,Mme CL. DESMONTILS,2875,False,False,(25)Da4aDa6a7a3a(24)1a1a,25.0,31.0
9,11,LINDA DES LANDES,Mlle NOEMIE HARDY,2850,False,False,3a(25)8a6a6a7a6a9aDa2a,27.0,34.0


## 5. Overround (taux de prélèvement implicite)

In [11]:
# Overround = sum(implied_prob) per race — PMU typically ~1.25-1.28
conn.execute(f"""
    SELECT
        o.race_id,
        ra.hippodrome,
        o.odds_type,
        ROUND(SUM(o.implied_prob), 4) AS overround,
        COUNT(*)                      AS nb_cotes
    FROM odds o
    JOIN races ra ON ra.race_id = o.race_id
    WHERE ra.date = '{TODAY}'
    GROUP BY o.race_id, ra.hippodrome, o.odds_type
    ORDER BY o.race_id, o.odds_type
""").df()

,race_id,hippodrome,odds_type,overround,nb_cotes
0,20260226-R1-C1,CAGNES/MER,final,1.1865,14
1,20260226-R1-C1,CAGNES/MER,morning,1.1907,14
2,20260226-R1-C2,CAGNES/MER,final,1.1776,11
3,20260226-R1-C2,CAGNES/MER,morning,1.1939,11
4,20260226-R1-C3,CAGNES/MER,final,1.2008,8
...,...,...,...,...,...
59,20260226-R7-C5,LYON LA SOIE,final,1.2053,8
60,20260226-R7-C6,LYON LA SOIE,final,1.3980,16
61,20260226-R7-C6,LYON LA SOIE,morning,1.3953,16
62,20260226-R7-C7,LYON LA SOIE,final,1.2277,3


## 6. Analyse historique (60 jours)
Statistiques sur l'ensemble des données passées : hippodromes, taux de réussite des favoris, effet du drift des cotes.

In [12]:
# Hippodromes les plus actifs sur la période historique
conn.execute(f"""
    SELECT
        hippodrome,
        COUNT(DISTINCT date)       AS jours,
        COUNT(*)                   AS courses,
        ROUND(AVG(field_size), 1)  AS moy_partants
    FROM races
    WHERE date < '{TODAY}'
    GROUP BY hippodrome
    ORDER BY courses DESC
    LIMIT 15
""").df()

,hippodrome,jours,courses,moy_partants
0,VINCENNES,42,360,12.4
1,CAGNES/MER,18,140,11.3
2,MONS (GHLIN),13,81,10.5
3,SOLVALLA,8,72,10.1
4,LYON LA SOIE,8,64,11.9
5,LA CEPIERE,7,54,11.7
6,SON PARDO,11,52,10.8
7,WOLVEGA,8,48,8.4
8,MAUQUENCHY,6,48,12.8
9,ARGENTAN,6,47,14.7


In [13]:
# Taux de victoire par rang de cote du matin (60 jours)
conn.execute(f"""
    WITH ranked AS (
        SELECT
            ru.runner_id,
            ru.race_id,
            ru.finish_position,
            ROW_NUMBER() OVER (PARTITION BY ru.race_id ORDER BY o.decimal_odds) AS odds_rank
        FROM runners ru
        JOIN odds o  ON o.runner_id  = ru.runner_id AND o.odds_type = 'morning'
        JOIN races ra ON ra.race_id  = ru.race_id
        WHERE NOT ru.scratch AND ra.date < '{TODAY}'
    )
    SELECT
        odds_rank                                                               AS rang,
        COUNT(*)                                                                AS partants,
        COUNT(*) FILTER (WHERE finish_position = 1)                             AS victoires,
        ROUND(100.0 * COUNT(*) FILTER (WHERE finish_position = 1) / COUNT(*), 1) AS win_pct,
        ROUND(100.0 * COUNT(*) FILTER (WHERE finish_position <= 3) / COUNT(*), 1) AS place_pct
    FROM ranked
    WHERE finish_position IS NOT NULL AND odds_rank <= 10
    GROUP BY odds_rank
    ORDER BY odds_rank
""").df()

,rang,partants,victoires,win_pct,place_pct
0,1,1057,465,44.0,74.6
1,2,1006,242,24.1,60.5
2,3,990,157,15.9,51.3
3,4,935,85,9.1,42.6
4,5,927,75,8.1,35.7
5,6,933,51,5.5,30.7
6,7,887,38,4.3,23.4
7,8,828,39,4.7,20.7
8,9,725,18,2.5,14.8
9,10,631,16,2.5,12.8


In [14]:
# Effet du drift des cotes sur le taux de victoire (60 jours)
conn.execute(f"""
    WITH m AS (SELECT runner_id, decimal_odds AS o_matin FROM odds WHERE odds_type = 'morning'),
         f AS (SELECT runner_id, decimal_odds AS o_final FROM odds WHERE odds_type = 'final')
    SELECT
        CASE
            WHEN f.o_final <= m.o_matin * 0.75 THEN '1. Forte baisse  (≤ -25%)'
            WHEN f.o_final <  m.o_matin         THEN '2. Baisse modérée'
            WHEN f.o_final =  m.o_matin         THEN '3. Stable'
            WHEN f.o_final <= m.o_matin * 1.5   THEN '4. Hausse modérée'
            ELSE                                      '5. Forte hausse  (> +50%)'
        END AS drift_cat,
        COUNT(*) AS nb,
        ROUND(100.0 * COUNT(*) FILTER (WHERE ru.finish_position = 1) / COUNT(*), 1) AS win_pct,
        ROUND(100.0 * COUNT(*) FILTER (WHERE ru.finish_position <= 3) / COUNT(*), 1) AS place_pct
    FROM runners ru
    JOIN m ON m.runner_id = ru.runner_id
    JOIN f ON f.runner_id = ru.runner_id
    JOIN races ra ON ra.race_id = ru.race_id
    WHERE NOT ru.scratch AND ru.finish_position IS NOT NULL
      AND ra.date < '{TODAY}'
    GROUP BY 1
    ORDER BY 1
""").df()

,drift_cat,nb,win_pct,place_pct
0,1. Forte baisse (≤ -25%),1775,19.2,51.5
1,2. Baisse modérée,2011,16.5,45.7
2,3. Stable,378,15.3,41.0
3,4. Hausse modérée,3547,10.4,31.4
4,5. Forte hausse (> +50%),2845,3.8,17.1


In [15]:
# Top 15 chevaux par nombre de victoires (60 jours, min. 3 courses)
conn.execute(f"""
    SELECT
        ru.horse_name,
        COUNT(*) FILTER (WHERE ru.finish_position = 1)                         AS wins,
        COUNT(*) FILTER (WHERE ru.finish_position IS NOT NULL)                  AS starts,
        ROUND(
            100.0 * COUNT(*) FILTER (WHERE ru.finish_position = 1) /
            NULLIF(COUNT(*) FILTER (WHERE ru.finish_position IS NOT NULL), 0),
        1) AS win_pct
    FROM runners ru
    JOIN races ra ON ra.race_id = ru.race_id
    WHERE NOT ru.scratch AND ra.date < '{TODAY}'
    GROUP BY ru.horse_name
    HAVING starts >= 3
    ORDER BY wins DESC, win_pct DESC
    LIMIT 15
""").df()

,horse_name,wins,starts,win_pct
0,NIKITA DES CHARMES,4,4,100.0
1,KROONER D'HERIPRE,3,3,100.0
2,MANHATTAN TWIST,3,3,100.0
3,PAPILLON BOKO,3,3,100.0
4,REALITY MOM,3,3,100.0
5,MUNHOA VEDAQUAISE,3,3,100.0
6,JOUR ET NUIT,3,3,100.0
7,CHITCHAT,3,3,100.0
8,MERCI RS,3,3,100.0
9,LOQUIDY,3,3,100.0
